# Query Decomposition
> Break complex queries into simpler sub-questions for targeted retrieval.

`DecompositionTransformer` splits a multi-part query into focused
sub-questions. Each sub-question retrieves independently, and results
are combined to answer the original complex query.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.query import DecompositionTransformer
from anchor.models import QueryBundle

## Define a Decomposition Function
The decompose function takes a query string and returns a list of
sub-question strings. In production this calls an LLM.

In [ ]:
def mock_decompose(query: str) -> list:
    """Break a query into sub-questions based on its key terms."""
    words = query.split()[:3]
    return [f"Sub-question {i}: {part}" for i, part in enumerate(words)]

# Test the decomposition
subs = mock_decompose("What is context engineering?")
print(f"Decomposed into {len(subs)} sub-questions:")
for s in subs:
    print(f"  {s}")

## Create the Decomposition Transformer
Pass the decompose function to `DecompositionTransformer`.

In [ ]:
decomp = DecompositionTransformer(decompose_fn=mock_decompose)

print(f"Transformer: {type(decomp).__name__}")

## Transform a Query
`transform()` returns a list of `QueryBundle` objects, one per
sub-question.

In [ ]:
query = QueryBundle(query_str="What is context engineering?")
print(f"Original: {query.query_str}\n")

sub_queries = decomp.transform(query)

print(f"Decomposed into {len(sub_queries)} sub-queries:")
for i, sq in enumerate(sub_queries):
    print(f"  [{i}] {sq.query_str}")

## Realistic Decomposition Example
A more realistic decompose function would produce meaningful sub-questions.

In [ ]:
def realistic_decompose(query: str) -> list:
    """Simulate LLM decomposition of complex queries."""
    decompositions = {
        "Compare RAG and fine-tuning for production LLM apps": [
            "What is retrieval-augmented generation (RAG)?",
            "What is LLM fine-tuning?",
            "What are the trade-offs between RAG and fine-tuning?",
            "Which approach works better for production applications?",
        ],
        "How do vector databases handle scaling and consistency?": [
            "How do vector databases handle horizontal scaling?",
            "What consistency models do vector databases use?",
            "What are the trade-offs between scaling and consistency?",
        ],
    }
    return decompositions.get(query, [f"Sub: {query}"])

realistic_decomp = DecompositionTransformer(decompose_fn=realistic_decompose)

complex_query = QueryBundle(
    query_str="Compare RAG and fine-tuning for production LLM apps"
)
subs = realistic_decomp.transform(complex_query)

print(f"Original: {complex_query.query_str}\n")
print(f"Sub-questions:")
for i, sq in enumerate(subs):
    print(f"  {i + 1}. {sq.query_str}")

## Decomposition Retrieval Pattern
Each sub-question retrieves independently. Results are aggregated to
build a comprehensive answer.

In [ ]:
def mock_retrieve(query_str: str) -> list:
    base = hash(query_str) % 100
    return [f"doc-{base + i}" for i in range(2)]

all_docs = []
for sq in subs:
    docs = mock_retrieve(sq.query_str)
    all_docs.extend(docs)
    print(f"  Q: {sq.query_str[:50]}")
    print(f"    -> {docs}")

unique = list(dict.fromkeys(all_docs))
print(f"\nTotal: {len(all_docs)} docs, {len(unique)} unique")

## Key Takeaways
- `DecompositionTransformer` breaks complex queries into focused sub-questions
- Each sub-question retrieves independently for targeted results
- Results are aggregated and deduplicated across sub-questions
- Best for multi-part or comparison queries
- The decompose function can use an LLM or rule-based logic

**Next:** [Step-Back Prompting](04_step_back.ipynb)